# ImmunoWatch — Model Training Walkthrough

**Continuous AI monitoring for immunocompromised patients.**

This notebook walks end-to-end through the ImmunoWatch modelling pipeline:
clinical motivation → the deep-learning stack → data → preprocessing → the
personal-baseline **LSTM autoencoder** → a **FastAI** training demo → the
infection-risk **Temporal Transformer** → **federated learning** → **SHAP**
explainability → clinical interpretation.

> Runs against the project modules in `ml/`, `data/`, and `inference/`. Execute
> `python data/simulator.py` first so the patient CSVs exist.

## 1. Clinical motivation

500,000+ Americans live with severe immunocompromise (chemotherapy, organ
transplant, HIV). For them, a bacterial infection a healthy immune system clears
in days can be fatal in **hours**. Febrile neutropenia carries mortality of
~5–11% and far higher when antibiotics are delayed (IDSA, 2011/2023).

The problem is *timing*: fever is a **lagging** indicator. By the time core
temperature crosses 38.3 °C, the infection has typically progressed for 12–24h.
ImmunoWatch watches the **leading** indicators — autonomic stress (HRV) and
inflammatory fluid shift (impedance) — which move *before* temperature.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # import project modules from the repo root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import constants as C

print("PyTorch", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Archetypes:", list(C.PATIENT_ARCHETYPES))

## 2. The deep-learning stack — and why PyTorch is primary

| Library | Role here | Why |
|---|---|---|
| **PyTorch** | All production training/inference | Flexible custom architectures; research standard; clean ONNX→TFLite export |
| **Keras** | Prototyping comparison (below) | High-level API breadth |
| **FastAI** | Training-efficiency demo (§6) | Same model, fewer lines; built on PyTorch |
| **TFLite** | Deployment target (`ml/export.py`) | Microcontroller inference on the implant |
| **JAX** | Future work | Optimised federated training loops at scale |

Below: the **same** single-layer LSTM defined in PyTorch and in Keras, side by
side. PyTorch wins for this project because the autoencoder's
encoder→bottleneck→decoder and the Transformer's three output heads need explicit
control over the forward pass and tensor flow.

In [ ]:
# --- PyTorch: explicit module, full control over the forward pass ---
import torch.nn as nn

class TorchLSTM(nn.Module):
    def __init__(self, n_in=3, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_in, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return torch.sigmoid(self.head(out[:, -1]))

torch_model = TorchLSTM()
print(torch_model)
print("params:", sum(p.numel() for p in torch_model.parameters()))

In [ ]:
# --- Keras: the same model, high-level and concise (demonstration only) ---
# Requires `keras` (and a backend) installed; shown to contrast the two APIs.
try:
    import keras
    from keras import layers
    keras_model = keras.Sequential([
        layers.Input(shape=(120, 3)),
        layers.LSTM(64),
        layers.Dense(1, activation="sigmoid"),
    ])
    keras_model.summary()
except Exception as e:
    print("Keras demo skipped:", e)

## 3. Data exploration

The simulator emits 30 days of 1-minute biosignals per archetype with realistic
circadian/diurnal rhythms and four injected events spaced ≥5 days apart. Note the
**leading HRV decline** before each temperature rise — the signal the whole
system is built to catch.

In [ ]:
frames = {}
for pid in C.PATIENT_ARCHETYPES:
    path = C.PATIENT_DATA_DIR / f"{pid}.csv"
    if path.exists():
        frames[pid] = pd.read_csv(path, parse_dates=["timestamp"])
print({pid: len(df) for pid, df in frames.items()})
frames[list(frames)[0]].head()

In [ ]:
sensors = [("temp_c", "°C", "#ef4444"), ("hrv_rmssd_ms", "ms", "#3b82f6"), ("impedance_ohm", "Ω", "#10b981")]
fig, axes = plt.subplots(len(frames), 3, figsize=(16, 9), sharex="col")
for r, (pid, df) in enumerate(frames.items()):
    for c, (col, unit, color) in enumerate(sensors):
        ax = axes[r, c]
        ax.plot(df["timestamp"], df[col], color=color, lw=0.4)
        for _, ev in df[df["event_label"] != "normal"].groupby("event_label"):
            ax.axvspan(ev["timestamp"].iloc[0], ev["timestamp"].iloc[-1], color="orange", alpha=0.2)
        if c == 0: ax.set_ylabel(pid, fontsize=9)
        if r == 0: ax.set_title(f"{col} ({unit})")
fig.suptitle("Biosignals across archetypes (events shaded)")
fig.tight_layout(); plt.show()

In [ ]:
# Cross-sensor correlation for one patient (note temp <-> hrv anti-correlation during events)
df0 = frames[list(frames)[0]]
corr = df0[["temp_c", "hrv_rmssd_ms", "impedance_ohm"]].corr()
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(["temp", "hrv", "imp"]); ax.set_yticklabels(["temp", "hrv", "imp"])
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
fig.colorbar(im); plt.show()

## 4. Preprocessing

Order matters clinically: **artifact removal → gap handling → normalisation →
feature engineering**. Below, the Butterworth low-pass removes the single-minute
false-alarm spike while preserving the multi-hour fever ramp.

In [ ]:
from ml.preprocessing import BiosignalPreprocessor, MODEL_FEATURES

pid = list(C.PATIENT_ARCHETYPES)[0]
pre = BiosignalPreprocessor(pid)
clean, feat_names = pre.fit_transform()
print("model channels:", feat_names)
print("engineered features:", pre.engineered_features[:6], "...")
clean[["timestamp", "temp_c", "hrv_rmssd_ms", "impedance_ohm", *MODEL_FEATURES]].head()

In [ ]:
# Before/after the temperature low-pass around the false-alarm spike
raw = pd.read_csv(C.PATIENT_DATA_DIR / f"{pid}.csv", parse_dates=["timestamp"])
fa = raw[raw["event_label"] == "false_alarm"]
if len(fa):
    t0 = fa["timestamp"].iloc[0]
    win = (raw["timestamp"] > t0 - pd.Timedelta(hours=1)) & (raw["timestamp"] < t0 + pd.Timedelta(hours=1))
    cwin = (clean["timestamp"] > t0 - pd.Timedelta(hours=1)) & (clean["timestamp"] < t0 + pd.Timedelta(hours=1))
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(raw.loc[win, "timestamp"], raw.loc[win, "temp_c"], label="raw", color="#9ca3af")
    ax.plot(clean.loc[cwin, "timestamp"], clean.loc[cwin, "temp_c"], label="filtered", color="#ef4444")
    ax.set_title("Butterworth low-pass removes the 1-minute artifact"); ax.legend(); plt.show()

In [ ]:
# Lag-feature correlation heatmap (engineered features)
import seaborn as sns
eng = [c for c in pre.engineered_features if c.startswith("temp_c_lag") or c in
       ("temp_delta_from_baseline", "hrv_trend_slope_1h", "impedance_pct_change_2h")]
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(clean[eng].corr(), cmap="vlag", center=0, ax=ax)
ax.set_title("Engineered feature correlations"); plt.show()

## 5. Personal baseline — LSTM autoencoder (PyTorch)

Trained on days 1–14 of *healthy* data only. Reconstruction error spikes on
infection windows the model has never seen. The threshold is the 95th percentile
of normal reconstruction error — calibrated per patient.

In [ ]:
from ml.baseline import BaselineTrainer

trainer = BaselineTrainer(pid)
summary = trainer.train()   # trains, calibrates threshold, saves checkpoint + plot
summary

In [ ]:
# Reconstruction-error distribution: normal vs anomaly windows
from ml.preprocessing import make_windows
feats = clean[list(MODEL_FEATURES)].to_numpy("float32")
windows = make_windows(feats, C.BASELINE_WINDOW_MINUTES, stride=C.BASELINE_TRAIN_STRIDE)
errors = trainer._reconstruction_errors(windows)
real = np.isin(clean["event_label"].to_numpy(), ["infection", "neutropenic_crisis", "viral_mild"])
labels = np.array([int(real[i*C.BASELINE_TRAIN_STRIDE : i*C.BASELINE_TRAIN_STRIDE + C.BASELINE_WINDOW_MINUTES].any())
                   for i in range(len(windows))])
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(errors[labels == 0], bins=40, alpha=0.7, label="normal", color="#10b981")
ax.hist(errors[labels == 1], bins=40, alpha=0.7, label="anomaly", color="#ef4444")
ax.axvline(trainer.threshold, ls="--", color="#f59e0b", label="threshold")
ax.set_title("Reconstruction error: normal vs anomaly"); ax.legend(); plt.show()

## 6. FastAI training demo

The same reconstruction objective, expressed with a FastAI `Learner` in a handful
of lines — `lr_find` + `fit_one_cycle` replace the hand-written training loop.
This is a *breadth* demonstration; production training uses the explicit PyTorch
loop in `ml/baseline.py`.

In [ ]:
try:
    from fastai.data.core import DataLoaders
    from fastai.learner import Learner
    from torch.utils.data import TensorDataset, DataLoader
    from ml.baseline import LSTMAutoencoder

    X = torch.from_numpy(windows)
    ds = TensorDataset(X, X)                       # autoencoder: target == input
    n = len(ds); cut = int(0.85 * n)
    dls = DataLoaders(DataLoader(ds, bs=64, shuffle=True),
                      DataLoader(ds, bs=64))
    learn = Learner(dls, LSTMAutoencoder(), loss_func=lambda p, y: ((p - y) ** 2).mean())
    learn.fit_one_cycle(2, 1e-3)                    # vs. the ~80-line PyTorch loop
except Exception as e:
    print("FastAI demo skipped:", e)

## 7. Infection-risk Temporal Transformer

Multi-head self-attention over 6-hour windows, three output heads (risk,
severity, time-to-event). Split is **by time, never shuffled** — shuffling would
leak the future and inflate metrics.

In [ ]:
from ml.predictor import PredictorTrainer, build_global_dataset

ptrainer = PredictorTrainer()
metrics = ptrainer.train()   # trains on pooled, time-split windows; saves predictor.pt + metrics
metrics

In [ ]:
# Attention over a sample infection window — which timesteps does the model attend to?
from ml.predictor import build_patient_windows
ds = build_patient_windows(pid)
pos = np.where(ds.y_risk == 1)[0]
if len(pos):
    sample = torch.from_numpy(ds.x[pos[0]][None])
    m = ptrainer.model.eval()
    h = m.positional_encoding(m.input_projection(sample))
    layer = m.encoder.layers[0]
    attn_out, attn_w = layer.self_attn(h, h, h, need_weights=True, average_attn_weights=True)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(attn_w[0].mean(0).detach().numpy(), color="#3b82f6")
    ax.set_title("Mean attention weight across the 6-hour window (infection sample)")
    ax.set_xlabel("timestep (minutes)"); plt.show()

In [ ]:
# Confusion matrix + ROC on the held-out test set
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
_, _, test = build_global_dataset()
with torch.no_grad():
    probs = torch.sigmoid(ptrainer.model(torch.from_numpy(test.x))["risk_logit"]).numpy()
y = test.y_risk.astype(int); preds = (probs >= 0.5).astype(int)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(confusion_matrix(y, preds), annot=True, fmt="d", cmap="Blues", ax=ax[0],
            xticklabels=["normal", "infection"], yticklabels=["normal", "infection"])
ax[0].set_title("Confusion matrix")
if len(np.unique(y)) > 1:
    fpr, tpr, _ = roc_curve(y, probs)
    ax[1].plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.3f}", color="#3b82f6")
    ax[1].plot([0, 1], [0, 1], "--", color="#9ca3af"); ax[1].legend(); ax[1].set_title("ROC")
plt.show()

## 8. Federated learning (FedAvg)

Each patient "chip" trains locally; only weight updates are shared. The panels
compare federated vs. local-only validation loss per patient, and the table
reports the AUC generalisation gain.

In [ ]:
from ml.federated import FederatedSimulation
fed_summary = FederatedSimulation().run()   # writes reports/federated/federated_rounds.png
fed_summary

## 9. Explainability — SHAP

Per-sensor attribution turns a risk score into a clinical narrative. The waterfall
shows how each sensor pushes the prediction away from the background expectation.

In [ ]:
from inference.engine import InferenceEngine
engine = InferenceEngine()
if engine.explainer is not None:
    win = engine._scaled_window(pid, C.PREDICTOR_WINDOW_MINUTES) if len(engine.buffers[pid]) >= C.PREDICTOR_WINDOW_MINUTES else None
    if win is not None:
        shap_vals = engine.explainer.explain(win)
        print("SHAP (temp, impedance, hrv):", [round(v, 3) for v in shap_vals])
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.barh(["temp", "impedance", "hrv"], shap_vals,
                color=["#ef4444" if v >= 0 else "#10b981" for v in shap_vals])
        ax.axvline(0, color="#9ca3af"); ax.set_title("Per-sensor SHAP contribution"); plt.show()

## 10. Clinical interpretation — what a physician does next

Given a **CRITICAL** ImmunoWatch alert (combined risk ≥ 0.85, HRV the primary
SHAP driver, estimated ~6h to presentation), the neutropenic-fever response is:

1. **CBC with differential** — confirm/quantify neutropenia (ANC).
2. **Blood cultures** ×2 (peripheral + line) **before** antibiotics.
3. **Empiric broad-spectrum antibiotics within 60 minutes** (e.g. cefepime or
   piperacillin-tazobactam) per IDSA febrile-neutropenia guidance.
4. Source work-up (chest imaging, urinalysis, line assessment); escalate to
   in-patient management.

The value of ImmunoWatch is **lead time**: surfacing this 12–24h earlier — while
the patient is still afebrile — is the difference between an outpatient catch and
an ICU admission.

> Research/portfolio prototype on synthetic data. Not a medical device.